# 08 - Qiskit Validation of Small Defect Hamiltonians

## Aim of this notebook

In the previous notebooks, we studied hydrogen-like defects using a classical one-dimensional tight-binding model. The full 40-site Hamiltonian is solved classically because it is small enough for direct diagonalization.

In this notebook, we build a small representative defect Hamiltonian and map it to qubit operators. The goal is to compare exact classical results with Qiskit-compatible quantum estimates.

This notebook does not claim to simulate the full material on quantum hardware. Instead, it provides a proof-of-concept NISQ validation layer for selected small Hamiltonians inspired by the defect model.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector

ROOT = Path('..').resolve()
RESULTS_DIR = ROOT / 'results'
FIGURE_DIR = ROOT / 'figures'

RESULTS_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

## Select a Representative Defect Strength

Notebook 7 selected candidate cases for Qiskit validation. We use the weak spread case as a small-defect reference because it is physically meaningful and close to the optimized weak spread configuration.

In [ ]:
candidate_path = RESULTS_DIR / 'notebook7_candidate_cases_for_qiskit.csv'

if candidate_path.exists():
    candidate_cases = pd.read_csv(candidate_path)
    weak_spread = candidate_cases[candidate_cases['case'] == 'Weak spread defects'].iloc[0]
    defect_strength = float(weak_spread['defect_strength'])
else:
    candidate_cases = pd.DataFrame()
    defect_strength = 0.5

defect_strength

## Define a Small Qubit Hamiltonian

We use a two-qubit Hamiltonian:

```text
H = (a + V_defect) Z0 + b Z1 + c X0 X1 + d Y0 Y1
```

This is not the full 40-site material model. It is a small test Hamiltonian used to validate the quantum-computing workflow.

In [ ]:
a = 0.20
b = -0.10
c = -1.00
d = -1.00

hamiltonian = SparsePauliOp.from_list(
    [
        ('ZI', a + defect_strength),
        ('IZ', b),
        ('XX', c),
        ('YY', d),
    ]
)

hamiltonian

## Exact Classical Reference

Before using a Qiskit workflow, we diagonalize the small Hamiltonian exactly. The lowest eigenvalue is the exact ground-state reference.

In [ ]:
hamiltonian_matrix = hamiltonian.to_matrix()
exact_eigenvalues, exact_eigenvectors = np.linalg.eigh(hamiltonian_matrix)
exact_ground_energy = float(np.min(exact_eigenvalues).real)

exact_ground_energy, exact_eigenvalues

## Qiskit Ansatz Circuit

We prepare a simple two-qubit trial circuit with Ry rotations and one entangling CNOT.

In [ ]:
def build_ansatz(theta):
    """Build a two-qubit ansatz circuit."""
    circuit = QuantumCircuit(2)
    circuit.ry(theta[0], 0)
    circuit.ry(theta[1], 1)
    circuit.cx(0, 1)
    return circuit


initial_theta = np.array([0.3, -0.4])
ansatz = build_ansatz(initial_theta)
ansatz

## Save a Simple Circuit Diagram

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2.4))
ax.hlines([1.5, 0.5], xmin=0.1, xmax=5.8, color='black', linewidth=1)
ax.text(0.0, 1.5, 'q0', va='center', ha='right')
ax.text(0.0, 0.5, 'q1', va='center', ha='right')
ax.text(1.1, 1.5, 'Ry(theta0)', va='center', ha='center', bbox=dict(boxstyle='round', facecolor='white'))
ax.text(1.1, 0.5, 'Ry(theta1)', va='center', ha='center', bbox=dict(boxstyle='round', facecolor='white'))
ax.plot([3.2, 3.2], [0.5, 1.5], color='black')
ax.plot(3.2, 1.5, marker='o', color='black')
ax.text(3.2, 0.5, 'X', va='center', ha='center', bbox=dict(boxstyle='circle', facecolor='white'))
ax.set_xlim(-0.3, 6.0)
ax.set_ylim(0.0, 2.0)
ax.axis('off')
ax.set_title('Two-qubit Ry-CNOT ansatz')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook8_ansatz_circuit.png', dpi=200)
plt.show()

## Qiskit Statevector Expectation Value

For a fixed ansatz, we compute the expectation value:

```text
<psi(theta)|H|psi(theta)>
```

In [ ]:
def qiskit_statevector_energy(theta):
    circuit = build_ansatz(theta)
    state = Statevector.from_instruction(circuit)
    return float(np.real(state.expectation_value(hamiltonian)))


fixed_ansatz_energy = qiskit_statevector_energy(initial_theta)
fixed_ansatz_energy

## Simple VQE-Style Minimization

We minimize the Qiskit statevector expectation value over the ansatz parameters. This is a small VQE-style workflow.

In [ ]:
optimization = minimize(
    qiskit_statevector_energy,
    x0=initial_theta,
    method='COBYLA',
    options={'maxiter': 300, 'rhobeg': 0.5},
)

vqe_theta = optimization.x
vqe_energy = float(optimization.fun)

vqe_energy, vqe_theta, optimization.success

## Result Table

The comparison metric is absolute error versus the exact classical ground-state energy.

In [ ]:
results = pd.DataFrame(
    [
        {
            'method': 'Exact classical diagonalization',
            'energy_estimate': exact_ground_energy,
            'error_vs_exact': 0.0,
        },
        {
            'method': 'Qiskit fixed ansatz statevector',
            'energy_estimate': fixed_ansatz_energy,
            'error_vs_exact': abs(fixed_ansatz_energy - exact_ground_energy),
        },
        {
            'method': 'Qiskit VQE-style statevector',
            'energy_estimate': vqe_energy,
            'error_vs_exact': abs(vqe_energy - exact_ground_energy),
        },
    ]
)

results.to_csv(RESULTS_DIR / 'notebook8_qiskit_validation_results.csv', index=False)
results

## Energy and Error Comparison

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(results['method'], results['energy_estimate'])
plt.ylabel('Energy estimate')
plt.title('Notebook 8 energy comparison')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook8_energy_comparison.png', dpi=200)
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(results['method'], results['error_vs_exact'], color='tab:red')
plt.ylabel('Absolute error vs exact')
plt.title('Notebook 8 error comparison')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'notebook8_error_comparison.png', dpi=200)
plt.show()

## Scientific Interpretation

The exact classical diagonalization gives the reference ground-state energy for the small two-qubit Hamiltonian. The fixed ansatz provides a Qiskit-compatible expectation value, but it is not optimized and therefore may have a larger error.

The VQE-style minimization improves the ansatz parameters to reduce the energy. Agreement with the exact result indicates that this small Hamiltonian can be evaluated using a Qiskit-compatible hybrid quantum-classical workflow.

This is a proof-of-concept validation layer. It does not replace the classical 40-site simulations and it does not claim full material simulation on quantum hardware.

## Conclusion

In this notebook, we introduced a Qiskit-compatible validation layer for the hydrogen-like defect project. Instead of attempting to run the full 40-site tight-binding Hamiltonian on quantum hardware, we selected a small representative defect Hamiltonian and mapped it to qubit operators.

The exact classical result provides a reference value, while Qiskit-based expectation-value estimation and VQE-style minimization provide quantum-computing estimates. This comparison demonstrates how the classical defect-modeling workflow can be connected to hybrid quantum-classical simulation.

The result should be interpreted as a proof-of-concept NISQ validation step, not as a full quantum simulation of a real material.